# ChurnPredictor — Monitoring y Drift Detection

## Objetivo

Este notebook simula el monitoreo en producción del modelo de churn. El objetivo es detectar 
cuándo los datos de entrada cambian significativamente respecto al dataset de entrenamiento 
(data drift) o cuándo el rendimiento del modelo se degrada (model drift).

**Herramienta:** Evidently — framework open source para monitoreo de modelos de ML  
**Dataset de referencia:** X_train (datos con los que se entrenó el modelo)  
**Dataset de producción:** X_test simulando datos nuevos de producción

## 1. Importación de Librerías

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, ClassificationPreset
from evidently.metrics import DatasetDriftMetric, DataDriftTable

## 2. Carga de Datos

In [2]:
from pathlib import Path

DATA_PATH   = Path().resolve().parent / 'data' / 'processed'
MODELS_PATH = Path().resolve().parent / 'src' / 'models'

# Cargar splits guardados
X_train = pd.read_parquet(DATA_PATH / 'X_train.parquet')
X_test  = pd.read_parquet(DATA_PATH / 'X_test.parquet')
y_train = pd.read_parquet(DATA_PATH / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(DATA_PATH / 'y_test.parquet').squeeze()

print(f"Referencia (train): {X_train.shape[0]:,} filas")
print(f"Producción (test):  {X_test.shape[0]:,} filas")

Referencia (train): 4,922 filas
Producción (test):  1,055 filas


## 3. Cargar Modelo y Generar Predicciones

In [3]:
import joblib
import json

# Cargar pipeline y configuración
pipeline = joblib.load(MODELS_PATH / 'xgb_pipeline.joblib')
with open(MODELS_PATH / 'resultados_finales.json') as f:
    config = json.load(f)

THRESHOLD = config['threshold']

# Generar predicciones para el test set
y_proba_train = pipeline.predict_proba(X_train)[:, 1]
y_proba_test  = pipeline.predict_proba(X_test)[:, 1]
y_pred_train  = (y_proba_train >= THRESHOLD).astype(int)
y_pred_test   = (y_proba_test  >= THRESHOLD).astype(int)

print(f"Threshold: {THRESHOLD}")
print(f"Churn rate train: {y_pred_train.mean():.2%}")
print(f"Churn rate test:  {y_pred_test.mean():.2%}")

Threshold: 0.5719
Churn rate train: 34.01%
Churn rate test:  35.45%


## 4. Reporte de Data Drift

In [4]:
# Preparar datasets para Evidently
reference_data = X_train.copy()
reference_data['target'] = y_train.values

current_data = X_test.copy()
current_data['target'] = y_test.values

# Generar reporte de data drift
drift_report = Report(metrics=[
    DatasetDriftMetric(),
    DataDriftTable()
])

drift_report.run(
    reference_data=reference_data,
    current_data=current_data
)

# Guardar reporte HTML
REPORTS_PATH = Path().resolve().parent / 'src' / 'monitoring'
drift_report.save_html(str(REPORTS_PATH / 'drift_report.html'))

# Ver resultado resumido
drift_result = drift_report.as_dict()
dataset_drift = drift_result['metrics'][0]['result']
print(f"Dataset drift detectado: {dataset_drift['dataset_drift']}")
print(f"Features con drift: {dataset_drift['number_of_drifted_columns']} de {dataset_drift['number_of_columns']}")
print(f"Reporte guardado en: src/monitoring/drift_report.html")

Dataset drift detectado: False
Features con drift: 0 de 20
Reporte guardado en: src/monitoring/drift_report.html


> **Observación — Data Drift:**
> No se detectó drift en ninguna de las 20 features. Esto es esperado dado que 
> el dataset de referencia (train) y el de producción simulado (test) provienen 
> de la misma distribución original. En producción real, este reporte se ejecutaría 
> periódicamente comparando datos nuevos de clientes vs el dataset de entrenamiento — 
> si el perfil de los clientes cambia significativamente (nuevos planes, cambios 
> demográficos, nuevos competidores), el drift se activaría y señalizaría la necesidad 
> de reentrenar el modelo.

## 5. Reporte de Rendimiento del Modelo

In [5]:
# Preparar datasets con predicciones para Evidently
reference_data['prediction'] = y_pred_train
current_data['prediction']   = y_pred_test

# Generar reporte de clasificación
classification_report = Report(metrics=[
    ClassificationPreset()
])

classification_report.run(
    reference_data=reference_data,
    current_data=current_data,
    column_mapping=None
)

classification_report.save_html(str(REPORTS_PATH / 'classification_report.html'))

print("Reporte de clasificación guardado en: src/monitoring/classification_report.html")

Reporte de clasificación guardado en: src/monitoring/classification_report.html


> **Observación — Rendimiento del Modelo:**
> El reporte de clasificación compara las métricas del modelo entre el dataset de 
> referencia (train) y el de producción simulado (test). En producción real este 
> reporte se ejecutaría semanalmente o mensualmente para detectar degradación del 
> modelo. Si el Recall en producción cae por debajo de 0.70 o el F1 baja más de 
> 5 puntos respecto al baseline, se activaría una alerta de reentrenamiento.

## 6. Simulación de Drift — Escenario de Producción

In [7]:
# Simular drift — modificar la distribución de MonthlyCharges y tenure
np.random.seed(42)

drifted_data = X_test.copy()
drifted_data['MonthlyCharges'] = drifted_data['MonthlyCharges'] * 1.3 + np.random.normal(0, 5, len(drifted_data))
drifted_data['tenure'] = (drifted_data['tenure'] * 0.6).astype(int)
drifted_data['Contract'] = np.random.choice(
    ['Month-to-month', 'One year', 'Two year'],
    size=len(drifted_data),
    p=[0.85, 0.10, 0.05]
)

drifted_data['target'] = y_test.values

# Agregar predicciones para drifted_data
y_proba_drifted = pipeline.predict_proba(drifted_data.drop(columns=['target']))[:, 1]
drifted_data['prediction'] = (y_proba_drifted >= THRESHOLD).astype(int)

# Generar reporte de drift con datos modificados
drift_report_sim = Report(metrics=[
    DatasetDriftMetric(),
    DataDriftTable()
])

drift_report_sim.run(
    reference_data=reference_data,
    current_data=drifted_data
)

drift_report_sim.save_html(str(REPORTS_PATH / 'drift_report_simulated.html'))

drift_result_sim = drift_report_sim.as_dict()
dataset_drift_sim = drift_result_sim['metrics'][0]['result']
print(f"Dataset drift detectado: {dataset_drift_sim['dataset_drift']}")
print(f"Features con drift: {dataset_drift_sim['number_of_drifted_columns']} de {dataset_drift_sim['number_of_columns']}")

Dataset drift detectado: False
Features con drift: 3 de 21


> **Observación — Simulación de Drift:**
> Al modificar artificialmente `MonthlyCharges` (+30%), `tenure` (-40%) y `Contract` 
> (más clientes mes a mes), Evidently detectó drift en 3 de 21 features. Sin embargo 
> el dataset completo no se considera drifted porque no supera el umbral del 50% 
> de features con drift.
>
> En producción esto representaría una **alerta amarilla** — cambios en el perfil 
> de clientes que aún no justifican reentrenamiento inmediato pero sí monitoreo 
> frecuente. Si el número de features con drift supera el 50%, se activaría una 
> alerta de reentrenamiento automático via Prefect.